# Statistical Threshold Baseline

The simplest possible baseline: per-feature mean/std computed from normal training data; a window is flagged anomalous if its max per-feature z-score exceeds a calibrated threshold. No learned model at all — this exists purely as a sanity floor every other model should beat.

Trained STRICTLY on `normal` sessions only — motor_off and transition sessions are excluded entirely, so this model's job is simply: does this look like normal running, or not.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
 
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
Z_LIMIT = 3.0
WINDOW_SCORE_PERCENTILE = 99.0
FILE_ANOMALY_FRACTION_THRESHOLD = 0.10

OUTPUT_DIR = "pure_normal_statistical_baseline_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
FINAL_EVAL_DIR = os.path.join(OUTPUT_DIR, "final_model_evaluation")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, FINAL_EVAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
# ==========================================================
# CELL 3 — detect normal and anomaly files
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def normal_state_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    excluded = [
        "motor_off",
        "motor_being_on",
        "motor_being_off",
        "transition",
        "off_on_spikes",
    ]

    if any(word in path_text for word in excluded):
        return None

    if "/normal/" in path_text or stem.startswith("normal_") or stem == "normal":
        return "normal_running"

    return None


def anomaly_label_for(path):
    path_text = normalized_path(path)

    if "motor_being_on" in path_text:
        return "motor_being_on"
    if "motor_being_off" in path_text:
        return "motor_being_off"
    if "/motor_off/" in path_text or "motor_off" in Path(path_text).stem:
        return "motor_off"
    if "electrical fault" in path_text or "abnormal_current" in path_text:
        return "electrical_fault"
    if "mechanical fault" in path_text or "abnormal_vibration" in path_text:
        return "mechanical_fault"
    if "thermal fault" in path_text or "abnormal_temp" in path_text:
        return "thermal_fault"

    return None

In [ ]:
# ==========================================================
# CELL 4 — discover and read dataset files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )

    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    missing = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {path.name}: {missing}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean



In [ ]:
# ==========================================================
# CELL 5 — load only normal WITH-LOAD files for training
# ==========================================================
def is_normal_with_load(path):
    """
    Returns True only for normal files recorded with load.
    """
    path_text = (
        normalized_path(path)
        .replace(" ", "_")
        .replace("-", "_")
    )

    # Reject without-load normal files first.
    without_load_markers = [
        "without_load",
        "withoutload",
        "no_load",
        "noload",
        "unloaded",
    ]

    if any(
        marker in path_text
        for marker in without_load_markers
    ):
        return False

    # Accept with-load normal files.
    with_load_markers = [
        "with_load",
        "withload",
        "normal_load",
        "normal_loaded",
    ]

    return any(
        marker in path_text
        for marker in with_load_markers
    )


files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

healthy_sessions = []
fault_sessions = []
excluded_normal_files = []

for file_path in files:
    normal_state = normal_state_for(file_path)
    fault_label = anomaly_label_for(file_path)

    if normal_state is None and fault_label is None:
        continue

    # Exclude normal files recorded without load.
    if (
        normal_state is not None
        and not is_normal_with_load(file_path)
    ):
        excluded_normal_files.append(
            os.path.relpath(file_path, BASE_PATH)
        )
        continue

    try:
        df = read_one(file_path)

    except Exception as error:
        print(
            "SKIP:",
            file_path,
            "|",
            error,
        )
        continue

    record = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(
            file_path,
            BASE_PATH,
        ),
        "rows": len(df),
    }

    if normal_state is not None:
        record.update({
            "state": normal_state,
            "load_condition": "with_load",
            "binary_label": 0,
        })

        healthy_sessions.append(record)

    else:
        record.update({
            "fault_label": fault_label,
            "binary_label": 1,
        })

        fault_sessions.append(record)


if not healthy_sessions:
    raise RuntimeError(
        "No normal WITH-LOAD files were loaded. "
        "Check the filenames or folder names."
    )

if len(healthy_sessions) < N_SPLITS:
    raise RuntimeError(
        f"Only {len(healthy_sessions)} normal with-load files "
        f"were found, but N_SPLITS={N_SPLITS}."
    )

if not fault_sessions:
    raise RuntimeError(
        "No anomaly files were loaded."
    )


print("\nNORMAL WITH-LOAD FILES USED FOR TRAINING")

for session in healthy_sessions:
    print("  USED:", session["file"])


print("\nNORMAL FILES EXCLUDED")

for file_name in excluded_normal_files:
    print("  EXCLUDED:", file_name)


print("\nDATASET SUMMARY")
print(
    "Normal with-load training files:",
    len(healthy_sessions),
)
print(
    "Normal files excluded:",
    len(excluded_normal_files),
)
print(
    "Anomaly evaluation files:",
    len(fault_sessions),
)

In [ ]:
# ==========================================================
# CELL 6 — create fixed-length windows
# ==========================================================
def make_windows_from_array(X):
    starts = np.arange(
        0,
        len(X) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_windows_from_sessions(sessions):
    all_windows = []

    for session in sessions:
        windows, _ = make_windows_from_array(
            session["X"]
        )

        if windows is not None:
            all_windows.append(windows)

    if not all_windows:
        return np.empty(
            (0, WINDOW_LEN, N_FEATURES),
            dtype=np.float32,
        )

    return np.concatenate(
        all_windows,
        axis=0,
    )



In [ ]:
# ==========================================================
# CELL 7 — fit statistical baseline and control limits
# ==========================================================
def fit_statistical_baseline(X_normal):
    rows = X_normal.reshape(
        -1,
        N_FEATURES,
    )

    feature_mean = rows.mean(
        axis=0
    ).astype(np.float32)

    feature_std = rows.std(
        axis=0,
        ddof=0,
    ).astype(np.float32)

    feature_std = np.maximum(
        feature_std,
        1e-12,
    )

    lower_control_limit = (
        feature_mean
        - Z_LIMIT * feature_std
    )

    upper_control_limit = (
        feature_mean
        + Z_LIMIT * feature_std
    )

    return {
        "mean": feature_mean,
        "std": feature_std,
        "lower": lower_control_limit,
        "upper": upper_control_limit,
    }


def score_windows(model, X):
    z_scores = np.abs(
        (X - model["mean"])
        / model["std"]
    )

    maximum_z_score = z_scores.max(
        axis=(1, 2)
    ).astype(np.float32)

    mean_z_score = z_scores.mean(
        axis=(1, 2)
    ).astype(np.float32)

    worst_feature_ids = np.argmax(
        z_scores.mean(axis=1),
        axis=1,
    )

    return (
        maximum_z_score,
        mean_z_score,
        worst_feature_ids,
    )


def calibrate_threshold(model, X_normal):
    maximum_z_score, _, _ = score_windows(
        model,
        X_normal,
    )

    return float(
        np.percentile(
            maximum_z_score,
            WINDOW_SCORE_PERCENTILE,
        )
    )

In [ ]:
# ==========================================================
# CELL 8 — create file-level and window-level predictions
# ==========================================================
def aggregate_predictions(
    sessions,
    model,
    score_threshold,
):
    file_rows = []
    window_rows = []

    for session in sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        (
            maximum_z_score,
            mean_z_score,
            worst_feature_ids,
        ) = score_windows(
            model,
            windows,
        )

        flags = (
            maximum_z_score > score_threshold
        ).astype(np.int64)

        state = session.get(
            "state",
            session.get("fault_label", "unknown"),
        )

        true_label = int(session["binary_label"])
        anomaly_fraction = float(np.mean(flags))

        file_prediction = int(
            anomaly_fraction
            >= FILE_ANOMALY_FRACTION_THRESHOLD
        )

        file_rows.append({
            "file": session["file"],
            "state": state,
            "true_label": true_label,
            "pred_label": file_prediction,
            "windows": len(windows),
            "anomaly_windows": int(np.sum(flags)),
            "anomaly_fraction": anomaly_fraction,
            "mean_max_z": float(
                np.mean(maximum_z_score)
            ),
            "maximum_z": float(
                np.max(maximum_z_score)
            ),
            "score_threshold": score_threshold,
        })

        for window_id in range(len(windows)):
            window_rows.append({
                "file": session["file"],
                "state": state,
                "window": window_id,
                "start_row": int(starts[window_id]),
                "true_label": true_label,
                "pred_label": int(flags[window_id]),
                "maximum_absolute_z": float(
                    maximum_z_score[window_id]
                ),
                "mean_absolute_z": float(
                    mean_z_score[window_id]
                ),
                "score_threshold": score_threshold,
                "worst_feature": FEATURE_COLUMNS[
                    int(worst_feature_ids[window_id])
                ],
            })

    return (
        pd.DataFrame(file_rows),
        pd.DataFrame(window_rows),
    )


In [ ]:
# ==========================================================
# CELL 9 — build file-wise cross-validation table
# ==========================================================
all_sessions = healthy_sessions + fault_sessions

cv_table = pd.DataFrame({
    "label": [
        session["binary_label"]
        for session in all_sessions
    ],
    "group": [
        session["file"]
        for session in all_sessions
    ],
})


In [ ]:
# ==========================================================
# CELL 10 — five-fold file-wise cross-validation
# ==========================================================
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
cv_file_results = []
cv_window_results = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    train_sessions = [
        all_sessions[index]
        for index in train_index
    ]

    test_sessions = [
        all_sessions[index]
        for index in test_index
    ]

    normal_train_sessions = [
        session
        for session in train_sessions
        if session["binary_label"] == 0
    ]

    rng = np.random.default_rng(SEED + fold)
    shuffled_ids = rng.permutation(
        len(normal_train_sessions)
    )

    number_calibration_files = max(
        1,
        round(0.20 * len(shuffled_ids)),
    )

    calibration_ids = set(
        shuffled_ids[:number_calibration_files]
    )

    fit_sessions = [
        session
        for index, session in enumerate(normal_train_sessions)
        if index not in calibration_ids
    ]

    calibration_sessions = [
        session
        for index, session in enumerate(normal_train_sessions)
        if index in calibration_ids
    ]

    X_fit = build_windows_from_sessions(
        fit_sessions
    )

    X_calibration = build_windows_from_sessions(
        calibration_sessions
    )

    model = fit_statistical_baseline(
        X_fit
    )

    score_threshold = calibrate_threshold(
        model,
        X_calibration,
    )

    fold_file_df, fold_window_df = aggregate_predictions(
        test_sessions,
        model,
        score_threshold,
    )

    fold_file_df.insert(0, "fold", fold)
    fold_window_df.insert(0, "fold", fold)

    cv_file_results.append(fold_file_df)
    cv_window_results.append(fold_window_df)

    y_true = fold_window_df["true_label"].to_numpy()
    y_pred = fold_window_df["pred_label"].to_numpy()
    y_score = fold_window_df[
        "maximum_absolute_z"
    ].to_numpy()

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "roc_auc": (
            roc_auc_score(
                y_true,
                y_score,
            )
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
        "score_threshold": score_threshold,
    }

    cv_metrics.append(fold_metrics)

    print("\nFOLD", fold)
    print(pd.Series(fold_metrics))

In [ ]:
# ==========================================================
# CELL 11 — combined cross-validation results
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)

cv_file_results_df = pd.concat(
    cv_file_results,
    ignore_index=True,
)

cv_window_results_df = pd.concat(
    cv_window_results,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(CV_DIR, "cv_metrics.csv"),
    index=False,
)

cv_file_results_df.to_csv(
    os.path.join(CV_DIR, "cv_file_predictions.csv"),
    index=False,
)

cv_window_results_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"),
    index=False,
)

cv_y_true = cv_window_results_df[
    "true_label"
].to_numpy()

cv_y_pred = cv_window_results_df[
    "pred_label"
].to_numpy()

cv_y_score = cv_window_results_df[
    "maximum_absolute_z"
].to_numpy()

overall_cv_metrics = {
    "accuracy": accuracy_score(
        cv_y_true,
        cv_y_pred,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        cv_y_true,
        cv_y_pred,
    ),
    "precision": precision_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        cv_y_true,
        cv_y_score,
    ),
}

print("\nOVERALL CV RESULTS")
print(pd.Series(overall_cv_metrics))

print(classification_report(
    cv_y_true,
    cv_y_pred,
    target_names=["normal", "anomaly"],
    zero_division=0,
))

ConfusionMatrixDisplay(
    confusion_matrix(
        cv_y_true,
        cv_y_pred,
        labels=[0, 1],
    ),
    display_labels=[
        "normal window",
        "anomaly window",
    ],
).plot(values_format="d")

plt.tight_layout()
plt.savefig(
    os.path.join(CV_DIR, "confusion_matrix.png"),
    dpi=180,
)
plt.show()


In [ ]:
# ==========================================================
# CELL 12 — train final statistical baseline on all normal data
# ==========================================================
X_all_normal = build_windows_from_sessions(
    healthy_sessions
)

final_model = fit_statistical_baseline(
    X_all_normal
)

FINAL_SCORE_THRESHOLD = calibrate_threshold(
    final_model,
    X_all_normal,
)

print("Final score threshold:", FINAL_SCORE_THRESHOLD)



In [ ]:
# ==========================================================
# CELL 13 — final model evaluation
# ==========================================================
rng = np.random.default_rng(SEED)

number_normal_evaluation_files = max(
    1,
    round(0.20 * len(healthy_sessions)),
)

normal_ids = rng.choice(
    len(healthy_sessions),
    size=number_normal_evaluation_files,
    replace=False,
)

selected_normal_sessions = [
    healthy_sessions[int(index)]
    for index in normal_ids
]

fault_groups = {}

for session in fault_sessions:
    fault_groups.setdefault(
        session["fault_label"],
        [],
    ).append(session)

selected_fault_sessions = [
    group[int(rng.integers(len(group)))]
    for group in fault_groups.values()
]

evaluation_sessions = (
    selected_normal_sessions
    + selected_fault_sessions
)

final_file_df, final_window_df = aggregate_predictions(
    evaluation_sessions,
    final_model,
    FINAL_SCORE_THRESHOLD,
)

final_y_true = final_window_df[
    "true_label"
].to_numpy()

final_y_pred = final_window_df[
    "pred_label"
].to_numpy()

final_y_score = final_window_df[
    "maximum_absolute_z"
].to_numpy()

final_metrics = {
    "accuracy": accuracy_score(
        final_y_true,
        final_y_pred,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        final_y_true,
        final_y_pred,
    ),
    "normal_accuracy": float(
        np.mean(
            final_y_pred[final_y_true == 0] == 0
        )
    ),
    "anomaly_accuracy": float(
        np.mean(
            final_y_pred[final_y_true == 1] == 1
        )
    ),
    "precision": precision_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        final_y_true,
        final_y_score,
    ),
    "score_threshold": FINAL_SCORE_THRESHOLD,
}

print("\nFINAL RESULTS")
print(pd.Series(final_metrics))

print(classification_report(
    final_y_true,
    final_y_pred,
    target_names=["normal", "anomaly"],
    zero_division=0,
))

final_file_df.to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "file_predictions.csv",
    ),
    index=False,
)

final_window_df.to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "window_predictions.csv",
    ),
    index=False,
)

pd.DataFrame([final_metrics]).to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "metrics.csv",
    ),
    index=False,
)



In [ ]:
# ==========================================================
# CELL 14 — save model, threshold, and complete C header
# ==========================================================
MODEL_PATH = os.path.join(
    FINAL_DIR,
    "statistical_baseline.pkl",
)

THRESHOLD_PATH = os.path.join(
    FINAL_DIR,
    "statistical_threshold.npz",
)

HEADER_PATH = os.path.join(
    FINAL_DIR,
    "statistical_baseline_model.h",
)

joblib.dump(
    final_model,
    MODEL_PATH,
)

np.savez(
    THRESHOLD_PATH,
    score_threshold=np.float32(
        FINAL_SCORE_THRESHOLD
    ),
    file_anomaly_fraction_threshold=np.float32(
        FILE_ANOMALY_FRACTION_THRESHOLD
    ),
)

with open(HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef STATISTICAL_BASELINE_MODEL_H\n")
    file.write("#define STATISTICAL_BASELINE_MODEL_H\n\n")
    file.write("#include <math.h>\n\n")

    file.write(f"#define STAT_N_FEATURES {N_FEATURES}\n")
    file.write(f"#define STAT_WINDOW_LEN {WINDOW_LEN}\n")
    file.write(f"#define STAT_SCALE_DIVISOR {SCALE_DIVISOR:.1f}f\n")
    file.write(f"#define STAT_SCORE_THRESHOLD {FINAL_SCORE_THRESHOLD:.10f}f\n\n")

    file.write(
        "static const float STAT_MEAN[STAT_N_FEATURES] = {\n"
    )
    for value in final_model["mean"]:
        file.write(f"    {float(value):.10f}f,\n")
    file.write("};\n\n")

    file.write(
        "static const float STAT_STD[STAT_N_FEATURES] = {\n"
    )
    for value in final_model["std"]:
        file.write(f"    {float(value):.10f}f,\n")
    file.write("};\n\n")

    file.write(
        "static const float STAT_LOWER[STAT_N_FEATURES] = {\n"
    )
    for value in final_model["lower"]:
        file.write(f"    {float(value):.10f}f,\n")
    file.write("};\n\n")

    file.write(
        "static const float STAT_UPPER[STAT_N_FEATURES] = {\n"
    )
    for value in final_model["upper"]:
        file.write(f"    {float(value):.10f}f,\n")
    file.write("};\n\n")

    file.write(
        "static inline int statistical_baseline_predict(\n"
        "    const float raw_window[STAT_WINDOW_LEN][STAT_N_FEATURES],\n"
        "    float *maximum_z,\n"
        "    int *worst_feature)\n"
        "{\n"
        "    *maximum_z = 0.0f;\n"
        "    *worst_feature = 0;\n\n"
        "    for (int t = 0; t < STAT_WINDOW_LEN; t++)\n"
        "    {\n"
        "        for (int feature = 0; feature < STAT_N_FEATURES; feature++)\n"
        "        {\n"
        "            float physical =\n"
        "                raw_window[t][feature]\n"
        "                / STAT_SCALE_DIVISOR;\n\n"
        "            float z = fabsf(\n"
        "                (physical - STAT_MEAN[feature])\n"
        "                / STAT_STD[feature]\n"
        "            );\n\n"
        "            if (z > *maximum_z)\n"
        "            {\n"
        "                *maximum_z = z;\n"
        "                *worst_feature = feature;\n"
        "            }\n"
        "        }\n"
        "    }\n\n"
        "    return *maximum_z > STAT_SCORE_THRESHOLD;\n"
        "}\n\n"
        "#endif\n"
    )

print("Saved:", MODEL_PATH)
print("Saved:", THRESHOLD_PATH)
print("Saved:", HEADER_PATH)



In [ ]:
# ==========================================================
# CELL 15 — manual statistical baseline test
# ==========================================================
def manual_statistical_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    thresholds = np.load(THRESHOLD_PATH)

    score_threshold = float(
        thresholds["score_threshold"]
    )

    file_threshold = float(
        thresholds[
            "file_anomaly_fraction_threshold"
        ]
    )

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    (
        maximum_z_score,
        mean_z_score,
        worst_feature_ids,
    ) = score_windows(
        model,
        X_test,
    )

    flags = (
        maximum_z_score > score_threshold
    ).astype(np.int64)

    anomaly_fraction = float(
        np.mean(flags)
    )

    file_decision = (
        "ANOMALY"
        if anomaly_fraction >= file_threshold
        else "HEALTHY"
    )

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (
            starts + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,
        "maximum_absolute_z": maximum_z_score,
        "mean_absolute_z": mean_z_score,
        "score_threshold": score_threshold,
        "worst_feature": [
            FEATURE_COLUMNS[int(index)]
            for index in worst_feature_ids
        ],
        "decision": np.where(
            flags == 1,
            "anomaly",
            "normal",
        ),
        "is_anomaly": flags,
    })

    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(
        output_csv,
        index=False,
    )

    print("\n" + "=" * 60)
    print("MANUAL STATISTICAL BASELINE TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Normal windows:", int(np.sum(flags == 0)))
    print("Anomalous windows:", int(np.sum(flags == 1)))
    print("Anomaly fraction:", anomaly_fraction)
    print("File threshold:", file_threshold)
    print("FILE DECISION:", file_decision)
    print("Score threshold:", score_threshold)
    print("Saved CSV:", output_csv)

    print("\nWindow decisions:")
    print(result["decision"].value_counts())

    print("\nMost frequent worst features:")
    print(
        result["worst_feature"]
        .value_counts()
        .head(8)
    )

    plt.figure(figsize=(12, 4))
    plt.plot(
        result["time_sec"],
        result["maximum_absolute_z"],
    )
    plt.axhline(
        score_threshold,
        linestyle="--",
        label="Z-score threshold",
    )
    plt.xlabel("Time [s]")
    plt.ylabel("Maximum absolute Z-score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_score.png",
        ),
        dpi=150,
    )
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.step(
        result["time_sec"],
        result["is_anomaly"],
        where="mid",
    )
    plt.yticks(
        [0, 1],
        ["normal", "anomaly"],
    )
    plt.xlabel("Time [s]")
    plt.ylabel("Decision")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_timeline.png",
        ),
        dpi=150,
    )
    plt.show()

    return result


In [ ]:
# ==========================================================
# CELL 16 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_statistical_prediction(
            TEST_FILE
        )
    else:
        print("No file selected.")